In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Reload data fresh (so this notebook can run standalone)
ratings = pd.read_csv('../../data/ml-100k/u.data', sep='\t',
                       names=['user_id', 'item_id', 'rating', 'timestamp'])
users = pd.read_csv('../../data/ml-100k/u.user', sep='|',
                     names=['user_id', 'age', 'gender', 'occupation', 'zip_code'])

genre_cols = ['unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy',
              'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
              'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies = pd.read_csv('../../data/ml-100k/u.item', sep='|', encoding='latin-1',
                      names=['item_id', 'title', 'release_date', 'video_release_date',
                             'IMDb_URL'] + genre_cols)
movies_clean = movies.drop(columns=['release_date', 'video_release_date', 'IMDb_URL'])

# Build the training set: one row per rating, with user + movie features attached
df = ratings.merge(users, on='user_id').merge(movies_clean, on='item_id')
print(df.shape)
df.head()

(100000, 28)


,user_id,item_id,rating,timestamp,age,gender,occupation,zip_code,title,unknown,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,196,242,3,881250949,49,M,writer,55105,Kolya (1996),0,...,0,0,0,0,0,0,0,0,0,0
1,186,302,3,891717742,39,F,executive,00000,L.A. Confidential (1997),0,...,0,1,0,0,1,0,0,1,0,0
2,22,377,1,878887116,25,M,writer,40206,Heavyweights (1994),0,...,0,0,0,0,0,0,0,0,0,0
3,244,51,2,880606923,28,M,technician,80525,Legends of the Fall (1994),0,...,0,0,0,0,0,1,0,0,1,1
4,166,346,1,886397596,47,M,educator,55113,Jackie Brown (1997),0,...,0,0,0,0,0,0,0,0,0,0


Preparing features (X) and target (y) for the model

In [ ]:
# Encode categorical user features
df_encoded = pd.get_dummies(df, columns=['gender', 'occupation'], drop_first=True)

# Features: age, gender/occupation dummies, all 19 genre flags
# Drop columns that shouldn't be used as predictive features
feature_cols = [c for c in df_encoded.columns if c not in
                 ['user_id', 'item_id', 'rating', 'timestamp', 'title', 'zip_code']]

X = df_encoded[feature_cols]
y = df_encoded['rating']

print(X.shape, y.shape)
X.head()

(100000, 41) (100000,)


,age,unknown,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,...,occupation_marketing,occupation_none,occupation_other,occupation_programmer,occupation_retired,occupation_salesman,occupation_scientist,occupation_student,occupation_technician,occupation_writer
0,49,0,0,0,0,0,1,0,0,0,...,False,False,False,False,False,False,False,False,False,True
1,39,0,0,0,0,0,0,1,0,0,...,False,False,False,False,False,False,False,False,False,False
2,25,0,0,0,0,1,1,0,0,0,...,False,False,False,False,False,False,False,False,False,True
3,28,0,0,0,0,0,0,0,0,1,...,False,False,False,False,False,False,False,False,True,False
4,47,0,0,0,0,0,0,1,0,1,...,False,False,False,False,False,False,False,False,False,False


Train test and split the data into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(80000, 41) (20000, 41) (80000,) (20000,)


Train Random Forest Regressor model on the training data

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

Evaluate the model on the test data and print the RMSE

In [ ]:
y_pred = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f'Mean Absolute Error (MAE): {mae:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f}')

Mean Absolute Error (MAE): 0.8666
Root Mean Squared Error (RMSE): 1.0926


Checking Feature Importance of the trained model

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(10))

age                 0.326479
Action              0.041608
gender_M            0.039515
Comedy              0.038465
Sci-Fi              0.033581
Thriller            0.033192
Romance             0.031449
Adventure           0.030689
occupation_other    0.028514
Crime               0.025907
dtype: float64


1. Saving the trained model using joblib t0 disc.

In [ ]:
import joblib
import os


os.makedirs('../models', exist_ok=True)

# Save the Random Forest model
joblib.dump(rf_model, '../models/random_forest_model.pkl')


joblib.dump(list(X.columns), '../models/feature_columns.pkl')

print("Random Forest model and feature columns saved to ../models/ directory.")

Random Forest model and feature columns saved to ../models/ directory.
